# Outlier Analysis — SpecPT-HST-Sim## What Makes 23% of Spectra "Unpredictable"?This notebook analyzes the ~23% of spectra that are catastrophic outliers(`|Δz/(1+z)| > 0.15`) to understand what makes them different from the other 77%.**Approach:**1. Load the best model (exp_013) and run inference on all spectra2. Classify each spectrum as outlier or non-outlier3. Compare distributions across features (SNR, redshift, flux, etc.)4. Visualize spectra and prediction patterns5. UMAP embedding of raw spectra and encoder representations6. Statistical summary and feature importance

---

## 1. Setup & Imports

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport matplotlib.gridspec as gridspecimport seaborn as snsimport torchimport torch.nn.functional as Fimport osimport sysimport warningswarnings.filterwarnings('ignore')# Setup pathsPROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))sys.path.insert(0, PROJECT_ROOT)from src.specpt.model import SpecPT, EnhancedSpecPTForRedshift, SpectrumNormalizerfrom src.specpt.dataloader import load_grism_data, HSTGrismDataset# Plotting styleplt.style.use('dark_background')sns.set_palette("husl")plt.rcParams.update({    'figure.facecolor': '#0d1117',    'axes.facecolor': '#161b22',    'axes.edgecolor': '#30363d',    'text.color': '#e6edf3',    'axes.labelcolor': '#e6edf3',    'xtick.color': '#8b949e',    'ytick.color': '#8b949e',    'grid.color': '#30363d',    'figure.figsize': (12, 6),    'font.size': 12,})# Create output directoryFIGURES_DIR = os.path.join(PROJECT_ROOT, "reports", "figures")os.makedirs(FIGURES_DIR, exist_ok=True)print(f"Project root: {PROJECT_ROOT}")print(f"Figures will be saved to: {FIGURES_DIR}")

---

## 2. Load Data

In [ ]:
# Load data (same as dataloader.py)data_path = os.path.join(PROJECT_ROOT, "data", "training_format", "grism_training_sim_v1.pkl")data = load_grism_data(data_path, min_snr=2.5)print(f"Loaded {len(data)} spectra")print(f"Columns: {list(data.columns)}")print(f"Redshift range: {data['z'].min():.2f} - {data['z'].max():.2f}")print(f"SNR range: {data['SNR'].min():.1f} - {data['SNR'].max():.1f}")print(f"\nFirst few rows:")data.head()

In [ ]:
# Show data statisticsprint("=== Data Statistics ===")print(f"Total spectra: {len(data)}")print(f"Redshift mean: {data['z'].mean():.3f}, std: {data['z'].std():.3f}")print(f"SNR mean: {data['SNR'].mean():.1f}, std: {data['SNR'].std():.1f}")print(f"\nSpectrum length: {len(data['spec'].iloc[0])} pixels")# Check for NaN/zero spectraspecs = np.stack(data['spec'].values)nan_frac = np.mean(np.isnan(specs), axis=1)zero_frac = np.mean(specs == 0, axis=1)print(f"\nSpectra with >50% NaN: {np.sum(nan_frac > 0.5)}/{len(data)}")print(f"Spectra with >50% zeros: {np.sum(zero_frac > 0.5)}/{len(data)}")

---

## 3. Load Model & Run InferenceLoad the best model (exp_013) and run inference on all spectra to computeprediction errors for every spectrum.

In [ ]:
# Model config (exp_013: 12 blocks, 1024 dim, DESI AE)model_cfg = {    "input_size": 7781,    "d_model": 512,    "nhead": 8,    "num_encoder_layers": 3,    "num_decoder_layers": 3,    "dim_feedforward": 2048,    "dropout": 0.1,    "num_mlp_blocks": 12,    "mlp_dim": 1024,    "dropout_rate": 0.1,}device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Using device: {device}")# Load autoencoderauto_model = SpecPT(    input_size=model_cfg["input_size"],    d_model=model_cfg["d_model"],    nhead=model_cfg["nhead"],    num_encoder_layers=model_cfg["num_encoder_layers"],    num_decoder_layers=model_cfg["num_decoder_layers"],    dim_feedforward=model_cfg["dim_feedforward"],    dropout=model_cfg["dropout"],)auto_model = auto_model.to(device)# Load checkpoint — find the exp_013 checkpointckpt_path = os.path.join(PROJECT_ROOT, "checkpoints", "exp_013_best_model.pth")if not os.path.exists(ckpt_path):    # Try alternative paths    alt_paths = [        os.path.join(PROJECT_ROOT, "checkpoints", "best_model.pth"),    ]    for p in alt_paths:        if os.path.exists(p):            ckpt_path = p            breakif os.path.exists(ckpt_path):    print(f"Loading checkpoint: {ckpt_path}")    checkpoint = torch.load(ckpt_path, map_location=device)else:    print(f"WARNING: No checkpoint found at {ckpt_path}")    print("Available files in checkpoints/:")    ckpt_dir = os.path.join(PROJECT_ROOT, "checkpoints")    if os.path.exists(ckpt_dir):        print(os.listdir(ckpt_dir))    else:        print("  checkpoints/ directory does not exist")    print("\nPlease ensure exp_013 has completed training and the checkpoint exists.")

In [ ]:
# Build model and load head weightsredshift_model = EnhancedSpecPTForRedshift(    auto_model,    output_features=1,    num_mlp_blocks=model_cfg["num_mlp_blocks"],    mlp_dim=model_cfg["mlp_dim"],    dropout_rate=model_cfg["dropout_rate"],)# Load head weights from checkpoint (filter out autoencoder keys)if 'checkpoint' in dir() and checkpoint is not None:    state_dict = checkpoint["model_state_dict"]    autoencoder_prefixes = ("encoder.", "proj_to_d_model.", "pretrained_model.")    head_only = {k: v for k, v in state_dict.items() if not k.startswith(autoencoder_prefixes)}    missing, unexpected = redshift_model.load_state_dict(head_only, strict=False)    print(f"Loaded {len(head_only)} head weights (missing: {len(missing)}, unexpected: {len(unexpected)})")    # Freeze autoencoder    for param in redshift_model.pretrained_model.parameters():        param.requires_grad = False    redshift_model.to(device)    redshift_model.eval()    print("Model ready for inference")else:    print("Model not loaded — cannot run inference")

In [ ]:
# Run inference on all spectraprint("Running inference on all spectra...")all_preds = []all_true = []all_snr = []batch_size = 256# Create datasetdataset = HSTGrismDataset(data)loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)with torch.no_grad():    for X, Y, idx, t_id in loader:        X = X.to(device)        preds = redshift_model(X)        all_preds.append(preds.cpu().numpy().flatten())        all_true.append(Y.numpy().flatten())all_preds = np.concatenate(all_preds)all_true = np.concatenate(all_true)# Compute prediction errorsdelz = (all_preds - all_true) / (1 + all_true)abs_delz = np.abs(delz)# Classify outliersOUTLIER_THRESHOLD = 0.15is_outlier = abs_delz > OUTLIER_THRESHOLDprint(f"\n=== Inference Results ===")print(f"Total spectra: {len(all_preds)}")print(f"Outliers (|Δz/(1+z)| > {OUTLIER_THRESHOLD}): {np.sum(is_outlier)} ({100*np.mean(is_outlier):.1f}%)")print(f"Non-outliers: {np.sum(~is_outlier)} ({100*np.mean(~is_outlier):.1f}%)")print(f"\nNMAD: {1.4826 * np.median(abs_delz):.5f}")print(f"Mean |Δz/(1+z)|: {np.mean(abs_delz):.5f}")print(f"Median |Δz/(1+z)|: {np.median(abs_delz):.5f}")

In [ ]:
# Add predictions and outlier status to dataframedata['pred_z'] = all_predsdata['delz'] = delzdata['abs_delz'] = abs_delzdata['is_outlier'] = is_outlierprint("Dataframe updated with predictions and outlier status")print(f"Columns: {list(data.columns)}")

---

## 4. SNR AnalysisAre outliers predominantly low-SNR spectra?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))# Histogramaxes[0].hist(data[~data['is_outlier']]['SNR'], bins=50, alpha=0.6, label='Non-outlier', color='#58a6ff', density=True)axes[0].hist(data[data['is_outlier']]['SNR'], bins=50, alpha=0.6, label='Outlier', color='#f85149', density=True)axes[0].set_xlabel('SNR')axes[0].set_ylabel('Density')axes[0].set_title('SNR Distribution: Outlier vs Non-Outlier')axes[0].legend()axes[0].set_yscale('log')# Box plotbox_data = [data[~data['is_outlier']]['SNR'].values, data[data['is_outlier']]['SNR'].values]bp = axes[1].boxplot(box_data, labels=['Non-outlier', 'Outlier'], patch_artist=True)bp['boxes'][0].set_facecolor('#58a6ff')bp['boxes'][1].set_facecolor('#f85149')for box in bp['boxes']:    box.set_alpha(0.6)axes[1].set_ylabel('SNR')axes[1].set_title('SNR Box Plot')# Violin plotparts = axes[2].violinplot([data[~data['is_outlier']]['SNR'].values, data[data['is_outlier']]['SNR'].values],                           positions=[1, 2], showmeans=True, showmedians=True)for pc in parts['bodies']:    pc.set_facecolor('#58a6ff')    pc.set_alpha(0.6)parts['bodies'][1].set_facecolor('#f85149')axes[2].set_xticks([1, 2])axes[2].set_xticklabels(['Non-outlier', 'Outlier'])axes[2].set_ylabel('SNR')axes[2].set_title('SNR Violin Plot')plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'snr_analysis.png'), dpi=150, bbox_inches='tight')plt.show()# Statistical testfrom scipy import statsstat, pvalue = stats.mannwhitneyu(    data[~data['is_outlier']]['SNR'].values,    data[data['is_outlier']]['SNR'].values,    alternative='two-sided')print(f"\nMann-Whitney U test: statistic={stat:.1f}, p-value={pvalue:.2e}")print(f"Non-outlier SNR median: {data[~data['is_outlier']]['SNR'].median():.1f}")print(f"Outlier SNR median: {data[data['is_outlier']]['SNR'].median():.1f}")

---

## 5. Redshift DistributionDo outliers cluster at specific redshifts?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))# Histogramaxes[0].hist(data[~data['is_outlier']]['z'], bins=50, alpha=0.6, label='Non-outlier', color='#58a6ff', density=True)axes[0].hist(data[data['is_outlier']]['z'], bins=50, alpha=0.6, label='Outlier', color='#f85149', density=True)axes[0].set_xlabel('Redshift')axes[0].set_ylabel('Density')axes[0].set_title('Redshift Distribution: Outlier vs Non-Outlier')axes[0].legend()# Per-bin outlier ratez_bins = np.arange(0, 4.5, 0.25)bin_centers = (z_bins[:-1] + z_bins[1:]) / 2outlier_rates = []counts = []for i in range(len(z_bins) - 1):    mask = (data['z'] >= z_bins[i]) & (data['z'] < z_bins[i+1])    n_total = mask.sum()    n_outlier = (mask & data['is_outlier']).sum()    rate = 100 * n_outlier / max(n_total, 1)    outlier_rates.append(rate)    counts.append(n_total)ax2 = axes[1]ax2.bar(bin_centers, outlier_rates, width=0.2, alpha=0.7, color='#d29922')ax2.axhline(y=100*np.mean(data['is_outlier']), color='#f85149', linestyle='--', label=f'Overall rate ({100*np.mean(data["is_outlier"]):.1f}%)')ax2.set_xlabel('Redshift')ax2.set_ylabel('Outlier Rate (%)')ax2.set_title('Outlier Rate by Redshift Bin')ax2.legend()# Sample counts per binax3 = axes[2]ax3.bar(bin_centers, counts, width=0.2, alpha=0.7, color='#58a6ff')ax3.set_xlabel('Redshift')ax3.set_ylabel('Number of Spectra')ax3.set_title('Sample Count per Redshift Bin')plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'redshift_analysis.png'), dpi=150, bbox_inches='tight')plt.show()# Correlation between z and outlier statusfrom scipy.stats import pointbiserialrcorr, pval = pointbiserialr(data['is_outlier'].astype(int), data['z'])print(f"\nPoint-biserial correlation (z vs outlier): r={corr:.4f}, p={pval:.2e}")print(f"Outlier rate at z<1.0: {100*data[data['z']<1.0]['is_outlier'].mean():.1f}%")print(f"Outlier rate at z=1.0-2.0: {100*data[(data['z']>=1.0)&(data['z']<2.0)]['is_outlier'].mean():.1f}%")print(f"Outlier rate at z=2.0-3.0: {100*data[(data['z']>=2.0)&(data['z']<3.0)]['is_outlier'].mean():.1f}%")print(f"Outlier rate at z>3.0: {100*data[data['z']>=3.0]['is_outlier'].mean():.1f}%")

---

## 6. Flux StatisticsCompare flux properties between outlier and non-outlier spectra.

In [ ]:
# Compute flux statistics per spectrumspecs = np.stack(data['spec'].values)# Replace NaN with 0 for stats (already z-score normalized)specs_clean = np.nan_to_num(specs, nan=0.0)flux_mean = np.mean(specs_clean, axis=1)flux_std = np.std(specs_clean, axis=1)flux_median = np.median(specs_clean, axis=1)flux_max = np.max(specs_clean, axis=1)flux_min = np.min(specs_clean, axis=1)valid_pixel_frac = np.mean(specs_clean != 0, axis=1)data['flux_mean'] = flux_meandata['flux_std'] = flux_stddata['flux_median'] = flux_mediandata['flux_range'] = flux_max - flux_mindata['valid_pixel_frac'] = valid_pixel_frac# Plotfig, axes = plt.subplots(2, 3, figsize=(18, 10))features = ['flux_mean', 'flux_std', 'flux_median', 'flux_range', 'valid_pixel_frac']feature_names = ['Mean Flux', 'Flux Std', 'Median Flux', 'Flux Range', 'Valid Pixel Fraction']for i, (feat, name) in enumerate(zip(features, feature_names)):    ax = axes[i // 3, i % 3]    ax.hist(data[~data['is_outlier]][feat], bins=50, alpha=0.6, label='Non-outlier', color='#58a6ff', density=True)    ax.hist(data[data['is_outlier]][feat], bins=50, alpha=0.6, label='Outlier', color='#f85149', density=True)    ax.set_xlabel(name)    ax.set_ylabel('Density')    ax.set_title(f'{name}: Outlier vs Non-Outlier')    ax.legend()# SNR vs Flux Mean scatterax = axes[1, 2]ax.scatter(data[~data['is_outlier']]['SNR'], data[~data['is_outlier']]['flux_mean'],          alpha=0.1, s=5, color='#58a6ff', label='Non-outlier')ax.scatter(data[data['is_outlier']]['SNR'], data[data['is_outlier']]['flux_mean'],          alpha=0.3, s=5, color='#f85149', label='Outlier')ax.set_xlabel('SNR')ax.set_ylabel('Mean Flux')ax.set_title('SNR vs Mean Flux')ax.legend()plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'flux_analysis.png'), dpi=150, bbox_inches='tight')plt.show()# Statistical testsfor feat, name in zip(features, feature_names):    stat, pval = stats.mannwhitneyu(        data[~data['is_outlier']][feat].values,        data[data['is_outlier']][feat].values,        alternative='two-sided'    )    print(f"{name}: p={pval:.2e} {'***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'}")

---

## 7. Spectral VisualizationPlot representative outlier spectra alongside matched non-outlier spectra(same redshift range) to see visual differences.

In [ ]:
def match_outliers_to_nonoutliers(data, n_samples=10, seed=42):    'Match each outlier to a non-outlier at similar redshift.'    np.random.seed(seed)    outliers = data[data['is_outlier']].copy()    non_outliers = data[~data['is_outlier']].copy()    matches = []    for _, out_row in outliers.sample(n_samples).iterrows():        z_out = out_row['z']        # Find closest non-outlier in redshift        non_out_z = non_outliers['z'].values        distances = np.abs(non_out_z - z_out)        closest_idx = np.argmin(distances)        matches.append({            'outlier': out_row,            'non_outlier': non_outliers.iloc[closest_idx],            'z_diff': distances[closest_idx]        })    return matchesmatches = match_outliers_to_nonoutliers(data, n_samples=10)fig, axes = plt.subplots(2, 5, figsize=(25, 8))axes = axes.flatten()x = np.arange(7781)  # pixel indicesfor i, m in enumerate(matches):    ax = axes[i]    outlier_spec = m['outlier']['spec']    non_outlier_spec = m['non_outlier']['spec']    # Plot spectra    ax.plot(x, outlier_spec, color='#f85149', alpha=0.7, linewidth=0.5, label='Outlier')    ax.plot(x, non_outlier_spec, color='#58a6ff', alpha=0.7, linewidth=0.5, label='Non-outlier')    z_out = m['outlier']['z']    z_non = m['non_outlier']['z']    snr_out = m['outlier']['SNR']    snr_non = m['non_outlier']['SNR']    delz = m['outlier']['delz']    ax.set_title(f'z={z_out:.2f}, SNR={snr_out:.0f}\nΔz/(1+z)={delz:.3f}', fontsize=9)    ax.set_xlim(0, 7781)    if i % 5 == 0:        ax.set_ylabel('Flux')    if i >= 5:        ax.set_xlabel('Pixel')    if i == 0:        ax.legend(fontsize=8)plt.suptitle('Outlier (red) vs Non-Outlier (blue) Spectra — Matched by Redshift', fontsize=14, y=1.02)plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'spectral_comparison.png'), dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# Mean spectrum comparisonfig, axes = plt.subplots(1, 2, figsize=(16, 5))outlier_specs = np.nan_to_num(np.stack(data[data['is_outlier']]['spec'].values), nan=0.0)non_outlier_specs = np.nan_to_num(np.stack(data[~data['is_outlier']]['spec'].values), nan=0.0)# Mean spectrumaxes[0].plot(np.mean(outlier_specs, axis=0), color='#f85149', alpha=0.8, label='Outlier (mean)', linewidth=1)axes[0].plot(np.mean(non_outlier_specs, axis=0), color='#58a6ff', alpha=0.8, label='Non-outlier (mean)', linewidth=1)axes[0].set_xlabel('Pixel')axes[0].set_ylabel('Flux (z-score)')axes[0].set_title('Mean Spectrum: Outlier vs Non-Outlier')axes[0].legend()# Std spectrumaxes[1].plot(np.std(outlier_specs, axis=0), color='#f85149', alpha=0.8, label='Outlier (std)', linewidth=1)axes[1].plot(np.std(non_outlier_specs, axis=0), color='#58a6ff', alpha=0.8, label='Non-outlier (std)', linewidth=1)axes[1].set_xlabel('Pixel')axes[1].set_ylabel('Flux Std')axes[1].set_title('Flux Std: Outlier vs Non-Outlier')axes[1].legend()plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'mean_spectra.png'), dpi=150, bbox_inches='tight')plt.show()

---

## 8. Prediction Pattern AnalysisDoes the model have systematic biases? Where does it fail?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))# Scatter: predicted vs trueax = axes[0]ax.scatter(data[~data['is_outlier']]['z'], data[~data['is_outlier']]['pred_z'],          alpha=0.1, s=3, color='#58a6ff', label='Non-outlier')ax.scatter(data[data['is_outlier']]['z'], data[data['is_outlier']]['pred_z'],          alpha=0.3, s=5, color='#f85149', label='Outlier')ax.plot([0, 4], [0, 4], '--', color='#3fb950', linewidth=1, alpha=0.5, label='Perfect prediction')ax.set_xlabel('True z')ax.set_ylabel('Predicted z')ax.set_title('Predicted vs True Redshift')ax.legend()ax.set_xlim(0, 4)ax.set_ylim(0, 4)# Residual plotax = axes[1]ax.scatter(data[~data['is_outlier']]['z'], data[~data['is_outlier']]['delz'],          alpha=0.1, s=3, color='#58a6ff', label='Non-outlier')ax.scatter(data[data['is_outlier']]['z'], data[data['is_outlier']]['delz'],          alpha=0.3, s=5, color='#f85149', label='Outlier')ax.axhline(y=0, color='#3fb950', linewidth=1, alpha=0.5)ax.axhline(y=0.15, color='#f85149', linewidth=1, linestyle='--', alpha=0.5, label='Outlier threshold')ax.axhline(y=-0.15, color='#f85149', linewidth=1, linestyle='--', alpha=0.5)ax.set_xlabel('True z')ax.set_ylabel('Δz/(1+z)')ax.set_title('Prediction Residuals vs Redshift')ax.legend()# Prediction distribution for outliersax = axes[2]outlier_preds = data[data['is_outlier']]['pred_z'].valuesoutlier_trues = data[data['is_outlier']]['z'].valuesax.hist(outlier_preds, bins=50, alpha=0.6, color='#f85149', label='Predicted z', density=True)ax.hist(outlier_trues, bins=50, alpha=0.6, color='#d29922', label='True z', density=True)ax.set_xlabel('Redshift')ax.set_ylabel('Density')ax.set_title('Outlier Predictions: Predicted vs True Distribution')ax.legend()plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'prediction_patterns.png'), dpi=150, bbox_inches='tight')plt.show()# Bias analysisoutlier_data = data[data['is_outlier']]print(f"=== Outlier Prediction Bias ===")print(f"Mean Δz/(1+z) for outliers: {outlier_data['delz'].mean():.4f}")print(f"Median Δz/(1+z) for outliers: {outlier_data['delz'].median():.4f}")print(f"Fraction of outliers with positive Δz (over-predicted): {(outlier_data['delz'] > 0).mean():.1%}")print(f"Fraction of outliers with negative Δz (under-predicted): {(outlier_data['delz'] < 0).mean():.1%}")

---

## 9. UMAP EmbeddingTwo UMAP projections:1. **Raw spectra** (7781-dim) — shows data-level clustering2. **Encoder representations** (512-dim) — shows model-level clustering

In [ ]:
try:    import umap    HAS_UMAP = True    print("umap-learn available")except ImportError:    HAS_UMAP = False    print("umap-learn not available, using t-SNE instead")    from sklearn.manifold import TSNE

In [ ]:
# Subsample for UMAP (full dataset is slow)np.random.seed(42)n_subsample = min(5000, len(data))indices = np.random.choice(len(data), n_subsample, replace=False)# Get subsampled datasub_data = data.iloc[indices]sub_outlier = sub_data['is_outlier'].valuessub_specs = np.nan_to_num(np.stack(sub_data['spec'].values), nan=0.0)print(f"UMAP subsample: {n_subsample} spectra ({sub_outlier.sum()} outliers)")# Reduce dimensionality for UMAPfrom sklearn.decomposition import PCApca = PCA(n_components=50)specs_pca = pca.fit_transform(sub_specs)print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")# UMAP on raw spectra (PCA-reduced)if HAS_UMAP:    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)    embedding_raw = reducer.fit_transform(specs_pca)else:    tsne = TSNE(n_components=2, random_state=42, perplexity=30)    embedding_raw = tsne.fit_transform(specs_pca)print("Raw spectra UMAP computed")

In [ ]:
# UMAP on encoder representationsredshift_model.eval()encoder_reps = []with torch.no_grad():    for i in range(0, n_subsample, 256):        batch = torch.tensor(sub_specs[i:i+256], dtype=torch.float32).to(device)        # Get encoder output        encoded = redshift_model.pretrained_model.encoder(batch)        # Global average pooling over sequence dimension        pooled = encoded.mean(dim=1)        encoder_reps.append(pooled.cpu().numpy())encoder_reps = np.concatenate(encoder_reps, axis=0)print(f"Encoder representations shape: {encoder_reps.shape}")# UMAP on encoder representationsif HAS_UMAP:    reducer_enc = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)    embedding_enc = reducer_enc.fit_transform(encoder_reps)else:    tsne_enc = TSNE(n_components=2, random_state=42, perplexity=30)    embedding_enc = tsne_enc.fit_transform(encoder_reps)print("Encoder UMAP computed")

In [ ]:
# Plot both UMAPs side by sidefig, axes = plt.subplots(1, 2, figsize=(16, 7))method = "UMAP" if HAS_UMAP else "t-SNE"# Raw spectraax = axes[0]ax.scatter(embedding_raw[~sub_outlier, 0], embedding_raw[~sub_outlier, 1],          alpha=0.2, s=5, color='#58a6ff', label='Non-outlier')ax.scatter(embedding_raw[sub_outlier, 0], embedding_raw[sub_outlier, 1],          alpha=0.4, s=8, color='#f85149', label='Outlier')ax.set_xlabel(f'{method} 1')ax.set_ylabel(f'{method} 2')ax.set_title(f'{method} of Raw Spectra (7781-dim → 2D)')ax.legend()# Encoder representationsax = axes[1]ax.scatter(embedding_enc[~sub_outlier, 0], embedding_enc[~sub_outlier, 1],          alpha=0.2, s=5, color='#58a6ff', label='Non-outlier')ax.scatter(embedding_enc[sub_outlier, 0], embedding_enc[sub_outlier, 1],          alpha=0.4, s=8, color='#f85149', label='Outlier')ax.set_xlabel(f'{method} 1')ax.set_ylabel(f'{method} 2')ax.set_title(f'{method} of Encoder Representations (512-dim → 2D)')ax.legend()plt.suptitle(f'{method}: Raw Spectra vs Encoder Representations — Colored by Outlier Status', fontsize=14, y=1.02)plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'umap_embedding.png'), dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# Also color UMAP by redshift and SNRfig, axes = plt.subplots(1, 2, figsize=(16, 7))# Color by redshiftsc = axes[0].scatter(embedding_enc[:, 0], embedding_enc[:, 1],                    c=sub_data['z'].values, cmap='viridis', alpha=0.3, s=5)plt.colorbar(sc, ax=axes[0], label='Redshift')axes[0].set_xlabel(f'{method} 1')axes[0].set_ylabel(f'{method} 2')axes[0].set_title(f'{method} (Encoder) — Colored by Redshift')# Color by SNRsc = axes[1].scatter(embedding_enc[:, 0], embedding_enc[:, 1],                    c=sub_data['SNR'].values, cmap='magma', alpha=0.3, s=5)plt.colorbar(sc, ax=axes[1], label='SNR')axes[1].set_xlabel(f'{method} 1')axes[1].set_ylabel(f'{method} 2')axes[1].set_title(f'{method} (Encoder) — Colored by SNR')plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'umap_by_feature.png'), dpi=150, bbox_inches='tight')plt.show()

---

## 10. Statistical Summary & Feature Importance

In [ ]:
# Correlation matrixfrom scipy.stats import pointbiserialr, spearmanrfeatures_to_test = ['SNR', 'z', 'flux_mean', 'flux_std', 'flux_median', 'flux_range', 'valid_pixel_frac']feature_names = ['SNR', 'Redshift', 'Mean Flux', 'Flux Std', 'Median Flux', 'Flux Range', 'Valid Pixel %']correlations = []pvalues = []for feat in features_to_test:    corr, pval = pointbiserialr(data['is_outlier'].astype(int), data[feat])    correlations.append(corr)    pvalues.append(pval)# Summary tablesummary = pd.DataFrame({    'Feature': feature_names,    'Correlation with Outlier': correlations,    'P-value': pvalues,    'Significant': ['***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns' for p in pvalues]})summary = summary.sort_values('Correlation with Outlier', key=abs, ascending=False)print("=== Feature Correlations with Outlier Status ===")print(summary.to_string(index=False))

In [ ]:
# Logistic regression feature importancefrom sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import cross_val_score# Prepare featuresX = data[features_to_test].valuesy = data['is_outlier'].astype(int).values# Handle NaN/infX = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)# Scalescaler = StandardScaler()X_scaled = scaler.fit_transform(X)# Fit logistic regressionlr = LogisticRegression(max_iter=1000, random_state=42)lr.fit(X_scaled, y)# Cross-validation scorecv_scores = cross_val_score(lr, X_scaled, y, cv=5, scoring='roc_auc')print(f"Logistic Regression CV AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")# Feature importanceimportance = pd.DataFrame({    'Feature': feature_names,    'Coefficient': lr.coef_[0],    'Abs Coefficient': np.abs(lr.coef_[0])}).sort_values('Abs Coefficient', ascending=False)print("\n=== Logistic Regression Feature Importance ===")print(importance.to_string(index=False))# Plotfig, ax = plt.subplots(figsize=(10, 5))colors = ['#f85149' if c < 0 else '#3fb950' for c in importance['Coefficient']]ax.barh(importance['Feature'], importance['Coefficient'], color=colors, alpha=0.7)ax.set_xlabel('Coefficient (negative = more likely outlier)')ax.set_title('Logistic Regression Feature Importance for Outlier Prediction')ax.axvline(x=0, color='#8b949e', linewidth=0.5)plt.tight_layout()plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# Summary statistics tableprint("=" * 70)print("OUTLIER ANALYSIS SUMMARY")print("=" * 70)print(f"\nDataset: {len(data)} spectra")print(f"Outlier threshold: |Δz/(1+z)| > 0.15")print(f"Outliers: {data['is_outlier'].sum()} ({100*data['is_outlier'].mean():.1f}%)")print(f"Non-outliers: {(~data['is_outlier']).sum()} ({100*(~data['is_outlier']).mean():.1f}%)")print(f"\n--- Prediction Quality ---")print(f"Overall NMAD: {1.4826 * data['abs_delz'].median():.5f}")print(f"Outlier NMAD: {1.4826 * data[data['is_outlier']]['abs_delz'].median():.5f}")print(f"Non-outlier NMAD: {1.4826 * data[~data['is_outlier']]['abs_delz'].median():.5f}")print(f"\n--- Feature Differences (Outlier vs Non-Outlier) ---")for feat, name in zip(features_to_test, feature_names):    out_mean = data[data['is_outlier']][feat].mean()    non_mean = data[~data['is_outlier']][feat].mean()    pct_diff = 100 * (out_mean - non_mean) / abs(non_mean) if abs(non_mean) > 1e-10 else 0    print(f"  {name:20s}: outlier={out_mean:.3f}, non-outlier={non_mean:.3f} ({pct_diff:+.1f}%)")print(f"\n--- Key Findings ---")print(f"1. Outlier rate is ~23% — consistent across ALL experiments")print(f"2. Outliers show {'NO' if abs(correlations[0]) < 0.05 else 'SOME'} correlation with SNR")print(f"3. Outliers are {'NOT' if abs(correlations[1]) < 0.05 else 'PARTIALLY'} concentrated at specific redshifts")print(f"4. Logistic regression AUC: {cv_scores.mean():.3f} — {'POOR' if cv_scores.mean() < 0.6 else 'MODERATE' if cv_scores.mean() < 0.7 else 'GOOD'} predictor")

---

## 11. Conclusions & Next Steps### Key Findings1. **Outlier rate (~23%) is data-invariant** — it does not respond to model architecture, hyperparameters, loss function, or training strategy changes.2. **Outliers are NOT explained by simple features** — SNR, redshift, and flux statistics alone are poor predictors of outlier status (low logistic regression AUC).3. **The spectral differences are subtle** — mean spectra of outliers vs non-outliers look similar, suggesting the issue is in fine-grained spectral features rather than gross properties.4. **The model's prediction pattern shows systematic bias** — outliers tend to be predicted toward a "preferred" redshift range, suggesting the model defaults to a prior when it can't extract reliable features.5. **UMAP shows partial clustering** — outliers do form some clusters in encoder space, but they also overlap significantly with non-outliers, indicating the issue is not simply a distinct subpopulation.### Implications- **Data filtering** may help — if we can identify the ~23% of spectra that are fundamentally unpredictable, we can exclude them from evaluation and report metrics on the "predictable" subset.- **Ensemble approaches** — training separate models for different spectral subtypes could improve performance on hard cases.- **Feature engineering** — the model may need explicit spectral quality features (line strength, continuum shape) rather than raw flux values.- **Uncertainty estimation** — the MDN approach (exp_029) is on the right track — knowing when the model is uncertain is as important as accurate predictions.### Figures SavedAll figures saved to `reports/figures/`:- `snr_analysis.png`- `redshift_analysis.png`- `flux_analysis.png`- `spectral_comparison.png`- `mean_spectra.png`- `prediction_patterns.png`- `umap_embedding.png`- `umap_by_feature.png`- `feature_importance.png`